In [1]:
!pip install "pymongo[srv]"


   ---------------------------------------- 0.0/822.0 kB ? eta -:--:--
   ---------------------------------------- 822.0/822.0 kB 8.8 MB/s eta 0:00:00

   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ----

In [1]:
from pymongo import MongoClient
from getpass import getpass
from urllib.parse import quote_plus

# Atlas에서 복사한 연결 문자열
uri_template = """
mongodb+srv://<db_username>:<db_password>@childtrafficcluster.bnz9rsn.mongodb.net/?appName=ChildTrafficCluster
""".strip()

username = input("MongoDB 사용자 이름: ")
password = getpass("MongoDB 비밀번호: ")

uri = (
    uri_template
    .replace("<db_username>", quote_plus(username))
    .replace("<db_password>", quote_plus(password))
)

client = MongoClient(
    uri,
    serverSelectionTimeoutMS=10000
)

try:
    client.admin.command("ping")
    print("MongoDB 연결 성공!")
except Exception as e:
    print("MongoDB 연결 실패:")
    print(e)


MongoDB 사용자 이름:  yudangeun_db_user
MongoDB 비밀번호:  ········


MongoDB 연결 실패:
SSL handshake failed: ac-jgisrit-shard-00-00.bnz9rsn.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1028) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-jgisrit-shard-00-02.bnz9rsn.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1028) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-jgisrit-shard-00-01.bnz9rsn.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1028) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 10.0s, Topology Description: <TopologyDescription id: 6ac4448bbdb923facc22d71a, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('ac-jgisrit-shard-00-00.bnz9rsn.mongodb.net', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('SSL handshake failed: ac-jgisrit-shard-00-00

In [3]:
import pandas as pd

# CSV 불러오기
main_df = pd.read_csv("../data_processed/final_main_49.csv")
speed_df = pd.read_csv("../data_processed/final_speed_bump_35.csv")

# MongoDB 데이터베이스/컬렉션 선택
db = client["child_traffic_safety"]

main_collection = db["final_main_49"]
speed_collection = db["final_speed_bump_35"]

# 기존 데이터가 있으면 중복 방지를 위해 비우기
main_collection.delete_many({})
speed_collection.delete_many({})

# DataFrame -> dict -> MongoDB 저장
main_records = main_df.to_dict("records")
speed_records = speed_df.to_dict("records")

main_collection.insert_many(main_records)
speed_collection.insert_many(speed_records)

print("메인 49개 업로드:", main_collection.count_documents({}))
print("과속방지턱 35개 업로드:", speed_collection.count_documents({}))

메인 49개 업로드: 49
과속방지턱 35개 업로드: 35
